# Task 3: Vectorized Convolutional Forward Pass via im2col Transformation from Scratch

**Objective:** Build high-performance convolutions by transforming sliding spatial windows into a single optimized matrix multiplication ($Y = W_{flat} \cdot X_{im2col} + b$).

### Mechanics

The standard 2D convolution requires 6 nested loops (batch, channels, height, width, kernel_h, kernel_w). By flattening all image patches into columns of a large matrix, we compute spatial filters efficiently using matrix operations.

In [1]:
import numpy as np
import torch
import torch.nn as nn

def get_im2col_indices(x_shape, kh, kw, padding=1, stride=1):
    # Extract dimensions
    N, C, H, W = x_shape
    # Compute output sizes
    out_h = int((H + 2 * padding - kh) / stride + 1)
    out_w = int((W + 2 * padding - kw) / stride + 1)
    
    # Generate indices matrices
    i0 = np.repeat(np.arange(kh), kw)
    i0 = np.tile(i0, C)
    i1 = stride * np.repeat(np.arange(out_h), out_w)
    j0 = np.tile(np.arange(kw), kh * C)
    j1 = stride * np.tile(np.arange(out_w), out_h)
    
    i = i0.reshape(-1, 1) + i1.reshape(1, -1)
    j = j0.reshape(-1, 1) + j1.reshape(1, -1)
    k = np.repeat(np.arange(C), kh * kw).reshape(-1, 1)
    return (k.astype(int), i.astype(int), j.astype(int))

def im2col_indices(x, kh, kw, padding=1, stride=1):
    # Zero-pad the input image
    x_padded = np.pad(x, ((0, 0), (0, 0), (padding, padding), (padding, padding)), mode='constant')
    k, i, j = get_im2col_indices(x.shape, kh, kw, padding, stride)
    
    cols = x_padded[:, k, i, j]
    C = x.shape[1]
    cols = cols.transpose(1, 2, 0).reshape(kh * kw * C, -1)
    return cols

class Conv2DScratch:
    def __init__(self, in_channels, out_channels, kernel_size, padding=1, stride=1):
        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kh = self.kw = kernel_size
        self.padding = padding
        self.stride = stride
        
        # Initialize parameters
        self.W = np.random.randn(out_channels, in_channels, self.kh, self.kw) * np.sqrt(2.0 / (in_channels * self.kh * self.kw))
        self.b = np.zeros((out_channels, 1))
        
    def forward(self, X):
        N, C, H, W = X.shape
        out_h = int((H + 2 * self.padding - self.kh) / self.stride + 1)
        out_w = int((W + 2 * self.padding - self.kw) / self.stride + 1)
        
        # Transform input array into columns
        X_col = im2col_indices(X, self.kh, self.kw, self.padding, self.stride)
        # Flatten filter weights
        W_row = self.W.reshape(self.out_channels, -1)
        
        # Single dot product execution
        out = np.dot(W_row, X_col) + self.b
        
        # Reshape to (out_channels, out_h, out_w, N) -> Transpose to (N, out_channels, out_h, out_w)
        out = out.reshape(self.out_channels, out_h, out_w, N)
        out = out.transpose(3, 0, 1, 2)
        return out

In [2]:
# Numerical verification against PyTorch
N, C, H, W = 4, 3, 32, 32
in_c, out_c = 3, 8
k_size = 3
padding = 1
stride = 2

# Generate sample image
x_np = np.random.randn(N, C, H, W).astype(np.float32)
x_tensor = torch.tensor(x_np)

# Instanciate Custom Conv
custom_conv = Conv2DScratch(in_channels=in_c, out_channels=out_c, kernel_size=k_size, padding=padding, stride=stride)

# Instantiate PyTorch equivalent Conv
pt_conv = nn.Conv2d(in_channels=in_c, out_channels=out_c, kernel_size=k_size, padding=padding, stride=stride)
pt_conv.weight.data = torch.tensor(custom_conv.W, dtype=torch.float32)
pt_conv.bias.data = torch.tensor(custom_conv.b.squeeze(), dtype=torch.float32)

# Compare outputs
custom_out = custom_conv.forward(x_np)
pt_out = pt_conv(x_tensor).detach().numpy()

difference = np.abs(custom_out - pt_out)
max_diff = np.max(difference)
mean_diff = np.mean(difference)

print(f"Maximum Absolute Discrepancy: {max_diff:.8e}")
print(f"Mean Absolute Discrepancy: {mean_diff:.8e}")
assert max_diff < 1e-5, "Failed: Outputs deviate significantly!"
print("Success: Custom im2col forward pass mathematically matches PyTorch Conv2D!")

Maximum Absolute Discrepancy: 1.08163650e-06
Mean Absolute Discrepancy: 9.84315285e-08
Success: Custom im2col forward pass mathematically matches PyTorch Conv2D!
